# 1. Импорт библиотек и настройки

In [1]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)
from pprint import pprint

# 2. Загрузка констант и динамики

In [2]:
const = {'Привлекаемые_средства': 380_000_000_000,
            'Ставка_купона_ОФЗ_ИН_л': 0.025,
            'Ставка_купона_ОФЗ_ПД': 0.1374,
            'Номинал_ОФЗ_ИН': 10_000,
            'Номинал_ОФЗ_ПД': 1000,
            'Количество_человек': 2_000_000,
            'НДФЛ': 0.13
        }
const

{'Привлекаемые_средства': 380000000000,
 'Ставка_купона_ОФЗ_ИН_л': 0.025,
 'Ставка_купона_ОФЗ_ПД': 0.1374,
 'Номинал_ОФЗ_ИН': 10000,
 'Номинал_ОФЗ_ПД': 1000,
 'Количество_человек': 2000000,
 'НДФЛ': 0.13}

In [3]:
from cbr_inflation import get_inflation, get_latest_inflation, get_latest_target
from datetime import date

inf = get_inflation()
# выбираем только последний год и последнее значение инфляции
inf_d = inf.copy()
inf_d['Год'] = inf_d['date'].dt.year
inf_d.rename(columns={
    'inflation': 'Инфляция',
    'target': 'Цель по инфляции'},inplace=True)
inf_d = inf_d[['Год','Инфляция']]
inf_d = inf_d.tail(1)
# автоматизируем продлжения ряда лет,и настраиваем вывод целовой инфляции
current_year = date.today().year
forecast_years = [ current_year + 1, current_year + 2]
inf2 = pd.DataFrame({
    'Год': forecast_years,
    'Инфляция': inf['target'].iloc[-1]})
inf_res = pd.concat([inf_d,inf2],ignore_index=True)

In [4]:
import requests
import pandas as pd

BASE_URL = "http://www.cbr.ru/dataservice"

# Параметры для депозитов физических лиц
PUBLICATION_ID = 18      # В целом по РФ (депозиты)
DATASET_ID = 37
# Ставки по вкладам физических лиц

# Запрашиваем данные
params = {
    "publicationId": PUBLICATION_ID,
    "y1": 2020,
    "y2": 2026,
    "i_ids": [DATASET_ID],
    "m1_ids": [2], # разрез в рублях
    "m2_ids": [7]  # до востребование больше года
}

response = requests.get(f"{BASE_URL}/dataEx", params=params)
data = response.json()
raw = data.get("RawData", [])

# Преобразуем в DataFrame
df = pd.DataFrame(raw)
# Переименуем колонки для удобства
df.rename(columns={
    'period': 'period_name',
    'date': 'date_str',
    'value': 'rate',
    'measure_1_id': 'currency_id',
    'measure_2_id': 'term_id',
    'period_id': 'period_id',
    'rowId': 'row_id'
}, inplace=True)
# Преобразуем дату в datetime
df['date'] = pd.to_datetime(df['date_str'], format='%d.%m.%Y')
df = df.sort_values('date').reset_index(drop=True)
# приводим в порядок следование столбцов
df = df[['date', 'rate', 'currency_id', 'term_id', 'period_name']]
sd = df.tail(1)
# создаем переменную имеющую единственную последнюю актуальную ставку
value = sd['rate'].iloc[0]

In [5]:
DEPOSIT_DECREMENT = 2.5 # коэфициент снижения 
base = value
inf_res['Ставка депозита'] = base - (DEPOSIT_DECREMENT/100) * inf_res.index
inf_res[['Инфляция','Ставка депозита']] = inf_res[['Инфляция','Ставка депозита']]/100 # переводим проценты в числа
inf_res

,Год,Инфляция,Ставка депозита
0,2026,0.0602,0.1284
1,2027,0.0400,0.1281
2,2028,0.0400,0.1279


# 3. ОФЗ ИН (л)

In [6]:
ofz_in_l =inf_res.copy()
ofz_in_l

,Год,Инфляция,Ставка депозита
0,2026,0.0602,0.1284
1,2027,0.0400,0.1281
2,2028,0.0400,0.1279


In [7]:
ofz_in_l['Привлекаемые средства'] = const['Привлекаемые_средства']
ofz_in_l['Количество человек'] = const['Количество_человек']
ofz_in_l['Ставка купона'] = const['Ставка_купона_ОФЗ_ИН_л']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона
0,2026,0.0602,0.1284,380000000000,2000000,0.0250
1,2027,0.0400,0.1281,380000000000,2000000,0.0250
2,2028,0.0400,0.1279,380000000000,2000000,0.0250


In [8]:
ofz_in_l['На руках у человека, руб'] = ofz_in_l['Привлекаемые средства'] / ofz_in_l['Количество человек']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб"
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000


In [9]:
ofz_in_l['Облигаций штук'] = ofz_in_l['На руках у человека, руб'] / const ['Номинал_ОФЗ_ИН']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000


In [10]:
ofz_in_l['Инфляционный множитель']= (1 + inf_res['Инфляция']).cumprod()
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467


In [11]:
ofz_in_l['Номинал после индексации'] = const['Номинал_ОФЗ_ИН']*ofz_in_l['Инфляционный множитель']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232


In [12]:
ofz_in_l['Номинал на начало'] = float(const['Номинал_ОФЗ_ИН'])
ofz_in_l.loc[ofz_in_l.index > 0, 'Номинал на начало'] = ofz_in_l['Номинал после индексации'].shift(1).fillna(const['Номинал_ОФЗ_ИН'])
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800


In [13]:
ofz_in_l['Индексация номинала'] = ofz_in_l['Номинал на начало'] * ofz_in_l['Инфляция']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432


In [14]:
ofz_in_l['Купон, руб'] = ofz_in_l['Номинал после индексации'] * ofz_in_l['Ставка купона']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб"
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000,265.0500
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800,275.6520
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432,286.6781


In [15]:
ofz_in_l['Доход без вычета'] = ofz_in_l['Купон, руб'] * ofz_in_l['Облигаций штук']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000,265.0500,5035.9500
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800,275.6520,5237.3880
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432,286.6781,5446.8835


In [16]:
ofz_in_l ['Налоговый вычет, руб']= ofz_in_l['На руках у человека, руб'] * const['НДФЛ']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета,"Налоговый вычет, руб"
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000,265.0500,5035.9500,24700.0000
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800,275.6520,5237.3880,24700.0000
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432,286.6781,5446.8835,24700.0000


In [17]:
ofz_in_l['Доход с вычетом'] = ofz_in_l['Налоговый вычет, руб'] + ofz_in_l['Доход без вычета']
ofz_in_l

,Год,Инфляция,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета,"Налоговый вычет, руб",Доход с вычетом
0,2026,0.0602,0.1284,380000000000,2000000,0.0250,190000.0000,19.0000,1.0602,10602.0000,10000.0000,602.0000,265.0500,5035.9500,24700.0000,29735.9500
1,2027,0.0400,0.1281,380000000000,2000000,0.0250,190000.0000,19.0000,1.1026,11026.0800,10602.0000,424.0800,275.6520,5237.3880,24700.0000,29937.3880
2,2028,0.0400,0.1279,380000000000,2000000,0.0250,190000.0000,19.0000,1.1467,11467.1232,11026.0800,441.0432,286.6781,5446.8835,24700.0000,30146.8835


In [18]:
ofz_in_l = ofz_in_l[['Год',
 'Привлекаемые средства',
 'Количество человек',
 'Инфляция',
 'Ставка купона',
 'На руках у человека, руб',
 'Облигаций штук',
 'Инфляционный множитель',
 'Номинал на начало',
 'Индексация номинала',
 'Номинал после индексации',
 'Купон, руб',
 'Доход без вычета',
 'Налоговый вычет, руб',
 'Доход с вычетом']]


# 4. ОФЗ ПД

In [19]:
ofz_pd = ofz_in_l [['Год']].copy()
ofz_pd['Привлекаемые средства'] = const ["Привлекаемые_средства"]
ofz_pd ["Количество человек"] = const ["Количество_человек"]
ofz_pd ["Ставка купона"] = const ['Ставка_купона_ОФЗ_ПД']
ofz_pd ["На руках у человека"] = ofz_in_l [["На руках у человека, руб"]].copy()
ofz_pd ['Облигаций, штук'] = ofz_pd ['На руках у человека'] / const ['Номинал_ОФЗ_ПД']
ofz_pd ['Купон'] = const ['Номинал_ОФЗ_ПД'] * const ['Ставка_купона_ОФЗ_ПД']
ofz_pd ["Доход, руб"] = ofz_pd ["Купон"] * ofz_pd ['Облигаций, штук']
ofz_pd ['НДФЛ'] = ofz_pd ['Доход, руб'] * const ['НДФЛ']
ofz_pd ['Доход после вычета налога'] = ofz_pd['Доход, руб'] - ofz_pd ['НДФЛ']
ofz_pd

,Год,Привлекаемые средства,Количество человек,Ставка купона,На руках у человека,"Облигаций, штук",Купон,"Доход, руб",НДФЛ,Доход после вычета налога
0,2026,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
1,2027,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
2,2028,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200


# 5. Депозит

In [20]:
depozit = ofz_in_l[['Год']].copy()
depozit ['Привлекаемые средства'] = const ['Привлекаемые_средства']
depozit ['Количество человек'] = const ['Количество_человек']
depozit ['На руках у человека'] = ofz_in_l ['На руках у человека, руб']
depozit ['Ставка депозита'] = inf_res['Инфляция'] 


In [21]:
depozit ['Коэфициент'] = (1 + inf_res['Инфляция'] / 12) **12
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент
0,2026,380000000000,2000000,190000.0000,0.0602,1.0619
1,2027,380000000000,2000000,190000.0000,0.0400,1.0407
2,2028,380000000000,2000000,190000.0000,0.0400,1.0407


In [22]:
depozit ['Накопленный множитель'] = depozit ['Коэфициент'].cumprod()
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель
0,2026,380000000000,2000000,190000.0000,0.0602,1.0619,1.0619
1,2027,380000000000,2000000,190000.0000,0.0400,1.0407,1.1052
2,2028,380000000000,2000000,190000.0000,0.0400,1.0407,1.1502


In [23]:
initial_amount = depozit['На руках у человека'].iloc[0]

In [24]:
depozit ['Сумма на конец года'] = initial_amount * depozit ['Накопленный множитель']
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года
0,2026,380000000000,2000000,190000.0000,0.0602,1.0619,1.0619,201758.9310
1,2027,380000000000,2000000,190000.0000,0.0400,1.0407,1.1052,209978.9011
2,2028,380000000000,2000000,190000.0000,0.0400,1.0407,1.1502,218533.7655


In [25]:
depozit ['Сумма на начало года'] = initial_amount
depozit.loc [depozit.index > 0, 'Сумма на начало года'] = depozit['Сумма на конец года']. shift (1)
# Более хорошая альтернатива:
# depozit['Сумма на начало года '] = depozit['Сумма на конец года'].shift(1).fillna(depozit['На руках у человека, руб'].iloc[0])
#depozit['Сумма начало'] = depozit['Сумма конец'].shift(1).fillna(initial_amount)
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года,Сумма на начало года
0,2026,380000000000,2000000,190000.0000,0.0602,1.0619,1.0619,201758.9310,190000.0000
1,2027,380000000000,2000000,190000.0000,0.0400,1.0407,1.1052,209978.9011,201758.9310
2,2028,380000000000,2000000,190000.0000,0.0400,1.0407,1.1502,218533.7655,209978.9011


In [26]:
depozit['Проценты']= depozit['Сумма на конец года'] - depozit['Сумма на начало года']
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года,Сумма на начало года,Проценты
0,2026,380000000000,2000000,190000.0000,0.0602,1.0619,1.0619,201758.9310,190000.0000,11758.9310
1,2027,380000000000,2000000,190000.0000,0.0400,1.0407,1.1052,209978.9011,201758.9310,8219.9701
2,2028,380000000000,2000000,190000.0000,0.0400,1.0407,1.1502,218533.7655,209978.9011,8554.8644


In [27]:
depozit = depozit [["Год", "Привлекаемые средства", 
                   "Количество человек", 
                   "На руках у человека",
                   "Ставка депозита",
                   "Коэфициент",
                   "Накопленный множитель",
                   "Сумма на начало года",
                   "Сумма на конец года",
                    "Проценты"]]
depozit          

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на начало года,Сумма на конец года,Проценты
0,2026,380000000000,2000000,190000.0000,0.0602,1.0619,1.0619,190000.0000,201758.9310,11758.9310
1,2027,380000000000,2000000,190000.0000,0.0400,1.0407,1.1052,201758.9310,209978.9011,8219.9701
2,2028,380000000000,2000000,190000.0000,0.0400,1.0407,1.1502,209978.9011,218533.7655,8554.8644


# 6. Доход за период (собрал без merge)

In [28]:
ofz_in_l_summary = pd.DataFrame({
    'Инструмент': ['ОФЗ ИН (л)'],
    'Номинал': [ofz_in_l['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [ofz_in_l ['Индексация номинала'].sum()*ofz_in_l['Облигаций штук'].iloc[0] +  ofz_in_l['Доход без вычета'].sum()],
})
ofz_in_l_summary ['Итоговая сумма'] = ofz_in_l['На руках у человека, руб'].iloc[0] + ofz_in_l_summary['Доход, до налогов'] + ofz_in_l ['Налоговый вычет, руб'].iloc [0] 
ofz_in_l_summary

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма
0,ОФЗ ИН (л),190000.0000,43595.5623,258295.5623


In [29]:
ofz_pd_summary = pd.DataFrame({
    'Инструмент': ['ОФЗ ПД'],
    'Номинал': [ofz_in_l ['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [ofz_pd['Доход, руб'].sum()],
    'Итоговая сумма': [ofz_in_l ['На руках у человека, руб'].iloc[0] + ofz_pd['Доход, руб'].sum()]})
ofz_pd_summary

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма
0,ОФЗ ПД,190000.0000,78318.0000,268318.0000


In [30]:
depozit_summary = pd.DataFrame({
    'Инструмент': ['Депозит'],
    'Номинал': [ofz_in_l ['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [depozit['Проценты'].sum()],
    'Итоговая сумма': [ofz_in_l ['На руках у человека, руб'].iloc[0] + depozit['Проценты'].sum()]})
depozit_summary

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма
0,Депозит,190000.0000,28533.7655,218533.7655


In [31]:
svodnay_dont_merge = pd.concat([ofz_in_l_summary, ofz_pd_summary,depozit_summary],ignore_index= True)
svodnay_dont_merge

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма
0,ОФЗ ИН (л),190000.0000,43595.5623,258295.5623
1,ОФЗ ПД,190000.0000,78318.0000,268318.0000
2,Депозит,190000.0000,28533.7655,218533.7655


In [32]:
inf_factor = (1 + inf_res['Инфляция']).prod()

In [33]:
svodnay_dont_merge['Очистка инфляции'] = svodnay_dont_merge['Итоговая сумма']/inf_factor
svodnay_dont_merge['Реальный доход'] = svodnay_dont_merge['Итоговая сумма'] - svodnay_dont_merge['Номинал']
svodnay_dont_merge

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма,Очистка инфляции,Реальный доход
0,ОФЗ ИН (л),190000.0000,43595.5623,258295.5623,225248.7898,68295.5623
1,ОФЗ ПД,190000.0000,78318.0000,268318.0000,233988.9398,78318.0000
2,Депозит,190000.0000,28533.7655,218533.7655,190574.1847,28533.7655


In [34]:
svodnay_dont_merge.loc[svodnay_dont_merge['Инструмент'] == 'ОФЗ ИН (л)', 'Очистка инфляции'] = None
# pohti_konec.loc[pohti_konec['Инструмент'] == 'ОФЗ ИН (л)', 'Реальный доход'] = None
svodnay_dont_merge

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма,Очистка инфляции,Реальный доход
0,ОФЗ ИН (л),190000.0000,43595.5623,258295.5623,NaN,68295.5623
1,ОФЗ ПД,190000.0000,78318.0000,268318.0000,233988.9398,78318.0000
2,Депозит,190000.0000,28533.7655,218533.7655,190574.1847,28533.7655


# Делаем таблицу с merge

In [ ]:
ofz_in_l_2 = ofz_in_l[['Год', 'На руках у человека, руб']].copy()
ofz_in_l_2.rename(columns={'На руках у человека, руб': 'Вложения'}, inplace=True)
ofz_in_l_2

,Год,Вложения
0,2026,190000.0000
1,2027,190000.0000
2,2028,190000.0000


In [ ]:
ofz_in_l_2['ОФЗ ИН доход'] = ofz_in_l['Индексация номинала']* ofz_in_l['Облигаций штук'] + ofz_in_l['Доход без вычета']
ofz_in_l_2

,Год,Вложения,ОФЗ ИН доход
0,2026,190000.0000,16473.9500
1,2027,190000.0000,13294.9080
2,2028,190000.0000,13826.7043


In [ ]:
ofz_in_l_2 = ofz_in_l_2.merge(ofz_pd[['Год', 'Доход, руб']], on='Год', how='left')
ofz_in_l_2.rename(columns={'Доход, руб': 'ОФЗ ПД доход'}, inplace=True)
ofz_in_l_2

,Год,Вложения,ОФЗ ИН доход,ОФЗ ПД доход
0,2026,190000.0000,16473.9500,26106.0000
1,2027,190000.0000,13294.9080,26106.0000
2,2028,190000.0000,13826.7043,26106.0000


In [ ]:
ofz_in_l_2 = ofz_in_l_2.merge(depozit[['Год', 'Проценты']], on='Год', how='left')
ofz_in_l_2.rename(columns={'Проценты': 'Депозит доход'}, inplace=True)
ofz_in_l_2

,Год,Вложения,ОФЗ ИН доход,ОФЗ ПД доход,Депозит доход
0,2026,190000.0000,16473.9500,26106.0000,11758.9310
1,2027,190000.0000,13294.9080,26106.0000,8219.9701
2,2028,190000.0000,13826.7043,26106.0000,8554.8644


In [ ]:
# строка итого
total_row = ofz_in_l_2[['ОФЗ ИН доход', 'ОФЗ ПД доход', 'Депозит доход']].sum()
total_row['Год'] = 'Итого'
total_row['Вложения'] = ofz_in_l_2['Вложения'].iloc[0]  # начальные вложения (одинаковы)
ofz_in_l_2 = pd.concat([ofz_in_l_2, pd.DataFrame([total_row])], ignore_index=True)
ofz_in_l_2

,Год,Вложения,ОФЗ ИН доход,ОФЗ ПД доход,Депозит доход
0,2026,190000.0000,16473.9500,26106.0000,11758.9310
1,2027,190000.0000,13294.9080,26106.0000,8219.9701
2,2028,190000.0000,13826.7043,26106.0000,8554.8644
3,Итого,190000.0000,43595.5623,78318.0000,28533.7655


# Делаем длинную 


In [ ]:
# 1. Убираем строку 'Итого' из df_income
df_income_without_total = ofz_in_l_2[ofz_in_l_2['Год'] != 'Итого']

# 2. Теперь применяем melt к данным без итогов
df_long = df_income_without_total.melt(
    id_vars=['Год', 'Вложения'],
    value_vars=['ОФЗ ИН доход', 'ОФЗ ПД доход', 'Депозит доход'],
    var_name='Инструмент',
    value_name='Доход'
)

# 3. Сортируем по году и инструменту
df_long = df_long.sort_values(['Год', 'Инструмент']).reset_index(drop=True)

df_long

,Год,Вложения,Инструмент,Доход
0,2026,190000.0000,Депозит доход,11758.9310
1,2026,190000.0000,ОФЗ ИН доход,16473.9500
2,2026,190000.0000,ОФЗ ПД доход,26106.0000
3,2027,190000.0000,Депозит доход,8219.9701
4,2027,190000.0000,ОФЗ ИН доход,13294.9080
5,2027,190000.0000,ОФЗ ПД доход,26106.0000
6,2028,190000.0000,Депозит доход,8554.8644
7,2028,190000.0000,ОФЗ ИН доход,13826.7043
8,2028,190000.0000,ОФЗ ПД доход,26106.0000


In [ ]:
test = pd.DataFrame()
test =  df_long.groupby('Инструмент', as_index=False)['Доход'].sum()
#agg(Итоговая_сумма= ('Доход', sum) 

test

,Инструмент,Доход
0,Депозит доход,28533.7655
1,ОФЗ ИН доход,43595.5623
2,ОФЗ ПД доход,78318.0000


# Нагрузка на государство


In [42]:
gos = inf_res.copy()
gos['Прибавка от инфляции'] = const['Привлекаемые_средства']*gos['Инфляция']
gos['Инфляционный множитель'] = (1+gos ['Инфляция']).cumprod()
gos['Тело долга'] = const['Привлекаемые_средства'] * gos['Инфляционный множитель']
gos['Расходы на купоны'] = gos['Тело долга'] * const['Ставка_купона_ОФЗ_ИН_л']
gos['Общие затраты'] = gos['Прибавка от инфляции']+gos['Расходы на купоны']
gos_1 = gos[['Год','Инфляция','Прибавка от инфляции','Тело долга','Расходы на купоны','Общие затраты']]
gos_1

,Год,Инфляция,Прибавка от инфляции,Тело долга,Расходы на купоны,Общие затраты
0,2026,0.0602,22876000000.0000,402876000000.0000,10071900000.0000,32947900000.0000
1,2027,0.0400,15200000000.0000,418991040000.0000,10474776000.0000,25674776000.0000
2,2028,0.0400,15200000000.0000,435750681600.0000,10893767040.0000,26093767040.0000


In [43]:
gos_1.loc['Итого'] = gos_1.sum(numeric_only=True)
gos_1= gos_1.drop('Итого', errors='ignore') 
gos_1

,Год,Инфляция,Прибавка от инфляции,Тело долга,Расходы на купоны,Общие затраты
0,2026.0000,0.0602,22876000000.0000,402876000000.0000,10071900000.0000,32947900000.0000
1,2027.0000,0.0400,15200000000.0000,418991040000.0000,10474776000.0000,25674776000.0000
2,2028.0000,0.0400,15200000000.0000,435750681600.0000,10893767040.0000,26093767040.0000


In [44]:
total_gos_1 = pd.DataFrame(gos_1.sum(numeric_only=True)).T
total_gos_1.index = ['Итого']

In [45]:
itog = pd.concat([gos_1,total_gos_1])
itog

,Год,Инфляция,Прибавка от инфляции,Тело долга,Расходы на купоны,Общие затраты
0,2026.0000,0.0602,22876000000.0000,402876000000.0000,10071900000.0000,32947900000.0000
1,2027.0000,0.0400,15200000000.0000,418991040000.0000,10474776000.0000,25674776000.0000
2,2028.0000,0.0400,15200000000.0000,435750681600.0000,10893767040.0000,26093767040.0000
Итого,6081.0000,0.1402,53276000000.0000,1257617721600.0000,31440443040.0000,84716443040.0000


In [46]:
gos_2 = gos[['Год']].copy()
gos_2['Тело долга'] = const['Привлекаемые_средства']
gos_2['Расходы на купон'] = gos_2['Тело долга'] * const['Ставка_купона_ОФЗ_ПД']
gos_2['Сумма возврата,НДФЛ'] = ofz_pd['НДФЛ']*const['Количество_человек']
gos_2['Итого при учете возврата НДФЛ'] = gos_2['Расходы на купон']-gos_2['Сумма возврата,НДФЛ']
gos_2

,Год,Тело долга,Расходы на купон,"Сумма возврата,НДФЛ",Итого при учете возврата НДФЛ
0,2026,380000000000,52212000000.0000,6787560000.0000,45424440000.0000
1,2027,380000000000,52212000000.0000,6787560000.0000,45424440000.0000
2,2028,380000000000,52212000000.0000,6787560000.0000,45424440000.0000


In [47]:
total_gos_2 = pd.DataFrame(gos_2.sum(numeric_only=True)).T
total_gos_2.index = ['Итого']
itog2 = pd.concat([gos_2,total_gos_2])
itog2

,Год,Тело долга,Расходы на купон,"Сумма возврата,НДФЛ",Итого при учете возврата НДФЛ
0,2026.0000,380000000000.0000,52212000000.0000,6787560000.0000,45424440000.0000
1,2027.0000,380000000000.0000,52212000000.0000,6787560000.0000,45424440000.0000
2,2028.0000,380000000000.0000,52212000000.0000,6787560000.0000,45424440000.0000
Итого,6081.0000,1140000000000.0000,156636000000.0000,20362680000.0000,136273320000.0000
